# 02 Missing Values

Review missing data across 95,000 employee records.

The goal is to recover reliable information, evaluate possible estimates, and preserve missing values when there is not enough evidence to fill them.

## 1. Missing Values Overview

Identify missing values, affected columns, and incomplete records.

In [1]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, f1_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [2]:
df = pd.read_csv("data/processed/01_duplicates_resolved.csv")

missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": df.isna().mean() * 100
})

missing_summary = (
    missing_summary[missing_summary["missing_count"] > 0]
    .sort_values("missing_pct", ascending=False)
)

display(missing_summary.round(2))
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
print(f"Total missing cells: {df.isna().sum().sum():,}")
print(f"Rows with missing values: {df.isna().any(axis=1).sum():,}")

,missing_count,missing_pct
ethnicity,23796,25.05
performance_review_score,14758,15.53
remote_status,10585,11.14
performance_rating,4726,4.97
education_level,4656,4.90
age,2877,3.03
gender,2865,3.02
years_experience,937,0.99
tenure_years,931,0.98


Rows: 95,000
Columns: 19
Total missing cells: 66,131
Rows with missing values: 50,334


### 1.1 Before and After Deduplication

Compare missing values before and after duplicate consolidation.

The differences reflect both removed duplicate records and recovered information.

In [3]:
raw_df = pd.read_csv("employee_pay_equity_messy.csv")

missing_comparison = pd.DataFrame({
    "before": raw_df.isna().sum(),
    "after": df.isna().sum()
})

missing_comparison["difference"] = (
    missing_comparison["after"] - missing_comparison["before"]
)

display(missing_comparison.sort_values("difference"))

,before,after,difference
ethnicity,24286,23796,-490
performance_review_score,15076,14758,-318
remote_status,10811,10585,-226
education_level,4752,4656,-96
age,2957,2877,-80
performance_rating,4803,4726,-77
gender,2926,2865,-61
tenure_years,969,931,-38
years_experience,969,937,-32
employee_id,0,0,0


## 2. Hidden Missing Values

Check whether missing data is stored as text, such as "Unknown", "N/A", or "None".

In [4]:
missing_indicators = {
    "missing", "n/a", "na", "none", "null",
    "unknown", "not provided", "not available", "-", "?"
}

results = []

for column in df.select_dtypes(include=["object", "string"]).columns:
    values = df[column].dropna().astype("string")
    normalized = values.str.strip().str.casefold()

    matches = values[normalized.isin(missing_indicators)]

    for value, count in matches.value_counts().items():
        results.append({
            "column": column,
            "value": value,
            "count": count
        })

hidden_missing = pd.DataFrame(results, columns=["column", "value", "count"])

display(hidden_missing.sort_values("count", ascending=False))

,column,value,count


## 3. Missingness Patterns

Check whether missing values occur together and whether patterns differ across data sources.

In [5]:
rating_missing = df["performance_rating"].isna()
score_missing = df["performance_review_score"].isna()

summary = pd.crosstab(
    rating_missing.map({True: "Missing", False: "Present"}),
    score_missing.map({True: "Missing", False: "Present"}),
    rownames=["Performance Rating"],
    colnames=["Review Score"]
)

display(summary)

Review Score,Missing,Present
Performance Rating,,
Missing,726,4000
Present,14032,76242


### 3.1 Performance Rating and Review Score

Compare review scores across performance rating categories to determine whether missing values can be recovered reliably.

In [6]:
relationship = (
    df.groupby("performance_rating", dropna=False)["performance_review_score"]
    .agg(["count", "min", "median", "max"])
    .round(2)
)

display(relationship)

,count,min,median,max
performance_rating,,,,
Below Expectations,6476,1.0,3.4,5.0
Exceeds Expectations,17754,1.0,3.4,5.0
Meets Expectations,35991,1.0,3.4,5.0
exceeds,7991,1.0,3.4,5.0
meets,8030,1.0,3.4,5.0
NaN,4000,1.0,3.4,5.0


**Finding:** All performance rating categories have the same median review score (3.4) and range (1.0 to 5.0).

**Decision:** Keep missing values in both columns unchanged. There is not enough evidence to recover them reliably.

### 3.2 Missing Value Relationships

Measure correlations between missing-value indicators to identify possible dependencies.

In [7]:
missing_columns = df.columns[df.isna().any()].tolist()

missing_correlation = (
    df[missing_columns]
    .isna()
    .astype(int)
    .corr()
    .round(3)
)

display(missing_correlation)

,age,gender,ethnicity,education_level,years_experience,tenure_years,remote_status,performance_rating,performance_review_score
age,1.000,0.004,0.005,-0.006,0.006,-0.004,-0.001,0.005,-0.006
gender,0.004,1.000,0.001,0.001,0.007,0.004,-0.001,0.003,-0.001
ethnicity,0.005,0.001,1.000,0.004,-0.004,-0.003,0.000,0.002,0.004
education_level,-0.006,0.001,0.004,1.000,-0.003,-0.008,0.000,0.003,-0.001
years_experience,0.006,0.007,-0.004,-0.003,1.000,-0.000,-0.002,0.010,-0.004
tenure_years,-0.004,0.004,-0.003,-0.008,-0.000,1.000,0.000,0.004,0.009
remote_status,-0.001,-0.001,0.000,0.000,-0.002,0.000,1.000,0.000,0.000
performance_rating,0.005,0.003,0.002,0.003,0.010,0.004,0.000,1.000,-0.001
performance_review_score,-0.006,-0.001,0.004,-0.001,-0.004,0.009,0.000,-0.001,1.000


### 3.3 Missing Values by Source

Compare missing-value percentages across the three HR data sources.

In [8]:
missing_by_source = (
    df.groupby("source_dataset")[missing_columns]
    .agg(lambda x: x.isna().mean() * 100)
    .T
    .round(2)
)

display(missing_by_source)

source_dataset,acquired_company_legacy,hris_export_2023,survey_response_2024
age,2.92,3.00,3.12
gender,3.00,3.01,3.04
ethnicity,24.55,25.13,25.15
education_level,5.05,4.79,5.00
years_experience,0.96,1.02,0.94
tenure_years,1.09,0.99,0.92
remote_status,10.82,11.22,11.17
performance_rating,4.91,4.99,4.98
performance_review_score,15.37,15.62,15.49


### 3.4 Findings

Missing-value correlations are close to zero, and missing rates are similar across all three sources.

No strong missingness patterns were identified. Each column will be handled individually.

## 4. Tenure Recovery

All hire dates are available, but 931 tenure values are missing.

I'll investigate whether these values can be calculated from hire dates.

In [9]:
print(f"Missing tenure values: {df['tenure_years'].isna().sum():,}")
print(f"Missing hire dates: {df['hire_date'].isna().sum():,}")

display(
    df.loc[
        df["tenure_years"].isna(),
        ["employee_id", "hire_date", "tenure_years"]
    ].head(20)
)

Missing tenure values: 931
Missing hire dates: 0


,employee_id,hire_date,tenure_years
95,E162984,2020-11-14,NaN
97,LEG-68420,06-16-2024,NaN
177,LEG-35355,2025/10/18,NaN
278,E117296,03-14-2021,NaN
320,E115528,2019-06-23,NaN
331,LEG-73711,25 Oct 2025,NaN
443,E140231,10-22-2023,NaN
522,E172777,11-27-2024,NaN
532,E152415,08 Jul 2024,NaN
788,E144799,02-25-2023,NaN


In [10]:
slash_mask = df["hire_date"].str.fullmatch(r"\d{2}/\d{2}/\d{4}", na=False)

hire_dates = pd.to_datetime(
    df["hire_date"].where(~slash_mask),
    format="mixed",
    errors="coerce",
    dayfirst=False
)

hire_dates.loc[slash_mask] = pd.to_datetime(
    df.loc[slash_mask, "hire_date"],
    format="%d/%m/%Y",
    errors="coerce"
)

print(f"Successfully parsed: {hire_dates.notna().sum():,}")
print(f"Failed to parse: {hire_dates.isna().sum():,}")

display(
    df.loc[
        hire_dates.isna(),
        ["employee_id", "hire_date"]
    ].head(20)
)

Successfully parsed: 95,000
Failed to parse: 0


,employee_id,hire_date


### 4.1 Validate Tenure Calculation

Calculate completed years of service using October 9, 2026, as the reference date.

This date was inferred from the existing tenure values and has not been independently verified.

In [11]:
reference_date = pd.Timestamp("2026-10-09")

calculated_tenure = (
    reference_date.year - hire_dates.dt.year
    - (
        (reference_date.month < hire_dates.dt.month)
        | (
            (reference_date.month == hire_dates.dt.month)
            & (reference_date.day < hire_dates.dt.day)
        )
    )
)

valid = df["tenure_years"].notna()

comparison = pd.DataFrame({
    "hire_date": hire_dates,
    "actual_tenure": df["tenure_years"],
    "calculated_tenure": calculated_tenure
}).loc[valid].copy()

comparison["difference"] = (
    comparison["actual_tenure"] - comparison["calculated_tenure"]
)

print(f"Exact matches: {(comparison['difference'] == 0).mean():.2%}")
print(f"Mismatched records: {(comparison['difference'] != 0).sum():,}")

display(
    comparison["difference"]
    .value_counts()
    .sort_index()
    .to_frame("count")
)

Exact matches: 100.00%
Mismatched records: 0


,count
difference,
0.0,94069


### 4.2 Recover Missing Tenure

The calculation matches all 94,069 existing tenure values.

I'll use the same calculation to recover 931 missing values while preserving the existing records.

In [12]:
df_clean = df.copy()

missing_tenure = df_clean["tenure_years"].isna()

df_clean.loc[missing_tenure, "tenure_years"] = calculated_tenure.loc[missing_tenure]

print(f"Recovered values: {missing_tenure.sum():,}")
print(f"Remaining missing tenure values: {df_clean['tenure_years'].isna().sum():,}")

assert df_clean["tenure_years"].notna().all()
assert df_clean.loc[~missing_tenure, "tenure_years"].equals(
    df.loc[~missing_tenure, "tenure_years"]
)

display(
    df_clean.loc[
        missing_tenure,
        ["employee_id", "hire_date", "tenure_years"]
    ].head(10)
)

Recovered values: 931
Remaining missing tenure values: 0


,employee_id,hire_date,tenure_years
95,E162984,2020-11-14,5.0
97,LEG-68420,06-16-2024,2.0
177,LEG-35355,2025/10/18,0.0
278,E117296,03-14-2021,5.0
320,E115528,2019-06-23,7.0
331,LEG-73711,25 Oct 2025,0.0
443,E140231,10-22-2023,2.0
522,E172777,11-27-2024,1.0
532,E152415,08 Jul 2024,2.0
788,E144799,02-25-2023,3.0


## 5. Age and Experience

Investigate the relationship between employee age and work experience to determine whether missing values can be recovered or estimated.

In [13]:
print(f"Missing age: {df_clean['age'].isna().sum():,}")
print(f"Missing experience: {df_clean['years_experience'].isna().sum():,}")

print(
    f"Correlation: "
    f"{df_clean[['age', 'years_experience']].corr().iloc[0, 1]:.3f}"
)

display(
    df_clean.groupby("age")["years_experience"]
    .agg(["count", "min", "median", "max"])
    .round(1)
)

Missing age: 2,877
Missing experience: 937
Correlation: 0.974


,count,min,median,max
age,,,,
20.0,4395,0.0,0.0,2.0
21.0,1157,0.0,0.0,3.0
22.0,1360,0.0,1.0,4.0
23.0,1539,0.0,2.0,5.0
24.0,1785,0.0,3.0,6.0
25.0,1997,1.0,4.0,7.0
26.0,2332,2.0,5.0,8.0
27.0,2655,3.0,6.0,9.0
28.0,2861,4.0,7.0,10.0


In [14]:
age_missing = df_clean["age"].isna()
experience_missing = df_clean["years_experience"].isna()

summary = pd.crosstab(
    age_missing.map({True: "Missing", False: "Present"}),
    experience_missing.map({True: "Missing", False: "Present"}),
    rownames=["Age"],
    colnames=["Experience"]
)

display(summary)

Experience,Missing,Present
Age,,
Missing,38,2839
Present,899,91224


### 5.1 Relationship Validation

Age and experience have a strong correlation of 0.974.

However, the relationship is not exact. Missing values cannot be recovered directly, but prediction methods can be evaluated.

In [15]:
valid = df_clean["age"].notna() & df_clean["years_experience"].notna()

age_experience_gap = (
    df_clean.loc[valid, "age"]
    - df_clean.loc[valid, "years_experience"]
)

display(
    age_experience_gap
    .value_counts()
    .sort_index()
    .rename_axis("age_minus_experience")
    .to_frame("count")
)

,count
age_minus_experience,
18.0,13139
19.0,12762
20.0,15562
21.0,12929
22.0,12543
23.0,12303
24.0,11928
25.0,16
26.0,11


**Finding:** Age and experience are strongly related, but the relationship does not allow exact recovery.

**Decision:** Keep missing values in both columns unchanged rather than introducing estimated data.

## 6. Missing Value Treatment

Evaluate different methods for handling missing data.

Reliable values may be recovered, while uncertain predictions will be stored separately from observed values.

### 6.1 Missing Value Flags

Create flags for the nine columns containing missing values.

These flags record the original missingness, even when values are later recovered.

In [16]:
missing_columns = df.columns[df.isna().any()].tolist()

for column in missing_columns:
    df_clean[f"{column}_is_missing"] = df[column].isna()

flag_columns = [f"{column}_is_missing" for column in missing_columns]

display(
    df_clean[flag_columns]
    .sum()
    .sort_values(ascending=False)
    .to_frame("flagged_records")
)

print(f"Missing flags created: {len(flag_columns)}")

,flagged_records
ethnicity_is_missing,23796
performance_review_score_is_missing,14758
remote_status_is_missing,10585
performance_rating_is_missing,4726
education_level_is_missing,4656
age_is_missing,2877
gender_is_missing,2865
years_experience_is_missing,937
tenure_years_is_missing,931


Missing flags created: 9


### 6.2 Age Estimation

Compare two methods for estimating missing ages:

- **Median:** Use the median age for employees with the same experience.
- **KNN:** Estimate age using experience and tenure.

Evaluate both methods using an 80/20 train-test split.

In [17]:
known = df_clean.dropna(subset=["age", "years_experience", "tenure_years"])

train, test = train_test_split(
    known,
    test_size=0.20,
    random_state=42
)

print(f"Training records: {len(train):,}")
print(f"Test records: {len(test):,}")

Training records: 72,979
Test records: 18,245


In [18]:
age_medians = train.groupby("years_experience")["age"].median()

median_pred = (
    test["years_experience"]
    .map(age_medians)
    .fillna(train["age"].median())
    .round()
    .astype(int)
)

In [19]:
features = ["years_experience", "tenure_years"]

knn = make_pipeline(
    StandardScaler(),
    KNeighborsRegressor(n_neighbors=25, weights="distance")
)

knn.fit(train[features], train["age"])

knn_pred = np.rint(knn.predict(test[features])).astype(int)

#### 6.2.1 Model Comparison

I'll compare both methods using three metrics:

- **MAE:** Average prediction error in years. Lower is better.
- **Exact Match:** Percentage of predictions that match the actual age.
- **Within ±1 Year:** Percentage of predictions within one year of the actual age.

In [20]:
results = []

for method, predictions in [
    ("Median", median_pred),
    ("KNN", knn_pred)
]:
    actual = test["age"]

    results.append({
        "method": method,
        "MAE": mean_absolute_error(actual, predictions),
        "Exact Match (%)": (actual == predictions).mean() * 100,
        "Within ±1 Year (%)": (abs(actual - predictions) <= 1).mean() * 100
    })

age_results = pd.DataFrame(results).set_index("method")

display(age_results.round(2))

,MAE,Exact Match (%),Within ±1 Year (%)
method,,,
Median,1.65,17.38,45.99
KNN,1.69,14.33,46.39


#### Findings

- **Median:** MAE of 1.65 years, with 17.38% exact matches.
- **KNN:** MAE of 1.69 years, with 14.33% exact matches.
- KNN performs slightly better within ±1 year (46.39% vs. 45.99%).

**Decision:** Median performs better overall and is simpler to maintain. I'll use it to estimate missing ages while keeping predictions separate from the original values.

#### 6.2.2 Predict Missing Ages

Median performed better overall, with an MAE of 1.65 years.

I'll use this method to estimate missing ages based on years of experience.

Predictions will be stored in `age_predicted`, keeping the original values unchanged.

In [21]:
age_medians = (
    df_clean.dropna(subset=["age", "years_experience"])
    .groupby("years_experience")["age"]
    .median()
)

df_clean["age_predicted"] = pd.Series(
    pd.NA, index=df_clean.index, dtype="Int64"
)

missing_age = df_clean["age"].isna()

df_clean.loc[missing_age, "age_predicted"] = (
    df_clean.loc[missing_age, "years_experience"]
    .map(age_medians)
    .round()
    .astype("Int64")
)

print(f"Missing ages: {missing_age.sum():,}")
print(f"Predicted ages: {df_clean['age_predicted'].notna().sum():,}")
print(f"Remaining missing predictions: {(missing_age & df_clean['age_predicted'].isna()).sum():,}")

display(
    df_clean.loc[
        missing_age,
        ["employee_id", "age", "years_experience", "age_predicted", "age_is_missing"]
    ].head(15)
)

Missing ages: 2,877
Predicted ages: 2,839
Remaining missing predictions: 38


,employee_id,age,years_experience,age_predicted,age_is_missing
40,E119181,NaN,3.0,25,True
82,E116556,NaN,4.0,26,True
161,E152539,NaN,26.0,46,True
221,E181338,NaN,15.0,36,True
259,E129683,NaN,19.0,40,True
261,E102530,NaN,16.0,37,True
308,E130572,NaN,22.0,43,True
315,E123996,NaN,20.0,40,True
329,E159477,NaN,19.0,40,True
335,E131079,NaN,9.0,30,True


#### 6.2.3 Validation

In [22]:
assert df_clean["age"].equals(df["age"])

assert df_clean.loc[
    df_clean["age"].notna(), "age_predicted"
].isna().all()

assert (
    df_clean["age_predicted"].notna()
    & ~df_clean["age_is_missing"]
).sum() == 0

print("Age prediction validation passed.")

Age prediction validation passed.


### 6.3 Experience Estimation

I'll compare Median and KNN to estimate missing work experience.

- **Median:** Use the median experience of employees of the same age.
- **KNN:** Find similar employees based on age and tenure.

I'll test both methods before choosing one.

In [23]:
known = df_clean.dropna(
    subset=["age", "years_experience", "tenure_years"]
)

train, test = train_test_split(
    known,
    test_size=0.20,
    random_state=42
)

print(f"Training records: {len(train):,}")
print(f"Test records: {len(test):,}")

Training records: 72,979
Test records: 18,245


In [24]:
experience_medians = (
    train.groupby("age")["years_experience"].median()
)

median_pred = (
    test["age"]
    .map(experience_medians)
    .fillna(train["years_experience"].median())
    .round()
    .astype(int)
)

In [25]:
features = ["age", "tenure_years"]

knn = make_pipeline(
    StandardScaler(),
    KNeighborsRegressor(n_neighbors=25, weights="distance")
)

knn.fit(train[features], train["years_experience"])

knn_pred = np.rint(
    knn.predict(test[features])
).astype(int)

#### 6.3.1 Model Comparison

I'll compare both methods using MAE, Exact Match, and Within ±1 Year accuracy.

In [26]:
results = []

for method, predictions in [
    ("Median", median_pred),
    ("KNN", knn_pred)
]:
    actual = test["years_experience"]

    results.append({
        "method": method,
        "MAE": mean_absolute_error(actual, predictions),
        "Exact Match (%)": (actual == predictions).mean() * 100,
        "Within ±1 Year (%)": (abs(actual - predictions) <= 1).mean() * 100
    })

experience_results = pd.DataFrame(results).set_index("method")

display(experience_results.round(2))

,MAE,Exact Match (%),Within ±1 Year (%)
method,,,
Median,1.64,17.60,45.77
KNN,1.60,20.22,48.44


#### 6.3.2 Predict Missing Experience

KNN performed better across all three metrics, with an MAE of 1.60 years.

I'll use KNN to estimate missing experience based on age and tenure.

Predictions will be stored separately, leaving the original values unchanged.

In [27]:
features = ["age", "tenure_years"]

known = df_clean.dropna(subset=features + ["years_experience"])

experience_knn = make_pipeline(
    StandardScaler(),
    KNeighborsRegressor(n_neighbors=25, weights="distance")
)

experience_knn.fit(known[features], known["years_experience"])

df_clean["years_experience_predicted"] = pd.Series(
    pd.NA, index=df_clean.index, dtype="Int64"
)

missing_experience = df_clean["years_experience"].isna()

predict_mask = (
    missing_experience
    & df_clean[features].notna().all(axis=1)
)

predictions = np.rint(
    experience_knn.predict(df_clean.loc[predict_mask, features])
).astype(int)

df_clean.loc[predict_mask, "years_experience_predicted"] = pd.Series(
    predictions,
    index=df_clean.index[predict_mask],
    dtype="Int64"
)

print(f"Missing experience: {missing_experience.sum():,}")
print(f"Predicted experience: {df_clean['years_experience_predicted'].notna().sum():,}")
print(f"Remaining missing predictions: {(missing_experience & df_clean['years_experience_predicted'].isna()).sum():,}")

display(
    df_clean.loc[
        missing_experience,
        [
            "employee_id",
            "age",
            "tenure_years",
            "years_experience",
            "years_experience_predicted",
            "years_experience_is_missing"
        ]
    ].head(15)
)

Missing experience: 937
Predicted experience: 899
Remaining missing predictions: 38


,employee_id,age,tenure_years,years_experience,years_experience_predicted,years_experience_is_missing
136,E160718,28.0,5.0,NaN,6,True
141,E180175,21.0,3.0,NaN,3,True
155,E192598,20.0,0.0,NaN,0,True
405,E143615,48.0,0.0,NaN,27,True
460,E140266,21.0,0.0,NaN,0,True
749,E179175,44.0,2.0,NaN,22,True
766,E194705,28.0,5.0,NaN,6,True
985,E147398,43.0,5.0,NaN,22,True
1105,E153332,28.0,0.0,NaN,7,True
1275,E142831,28.0,4.0,NaN,6,True


#### 6.3.3 Validation

In [28]:
assert df_clean["years_experience"].equals(df["years_experience"])

assert df_clean.loc[
    df_clean["years_experience"].notna(),
    "years_experience_predicted"
].isna().all()

assert (
    df_clean["years_experience_predicted"].notna()
    & ~df_clean["years_experience_is_missing"]
).sum() == 0

assert df_clean["years_experience_predicted"].dropna().between(0, 40).all()

print("Experience prediction validation passed.")

Experience prediction validation passed.


### 6.4 Performance Rating

The performance rating column contains different labels for the same categories.

I'll standardize these labels and convert them into three ordered levels:

- **1:** Below Expectations
- **2:** Meets Expectations
- **3:** Exceeds Expectations

The original values will remain unchanged.

In [29]:
display(
    df_clean["performance_rating"]
    .value_counts(dropna=False)
    .to_frame("count")
)

,count
performance_rating,
Meets Expectations,42617
Exceeds Expectations,21015
meets,9509
exceeds,9489
Below Expectations,7644
NaN,4726


In [30]:
rating_mapping = {
    "below expectations": "Below Expectations",
    "meets": "Meets Expectations",
    "meets expectations": "Meets Expectations",
    "exceeds": "Exceeds Expectations",
    "exceeds expectations": "Exceeds Expectations"
}

normalized = (
    df_clean["performance_rating"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

unknown = normalized.notna() & ~normalized.isin(rating_mapping)

assert not unknown.any(), "Unexpected performance rating categories found."

df_clean["performance_rating"] = normalized.map(rating_mapping).astype("string")

ordinal_mapping = {
    "Below Expectations": 1,
    "Meets Expectations": 2,
    "Exceeds Expectations": 3
}

df_clean["performance_rating_ordinal"] = (
    df_clean["performance_rating"]
    .map(ordinal_mapping)
    .astype("Int64")
)

display(
    df_clean[
        ["performance_rating", "performance_rating_ordinal"]
    ].value_counts(dropna=False).to_frame("count")
)

,,count
performance_rating,performance_rating_ordinal,
Meets Expectations,2,52126
Exceeds Expectations,3,30504
Below Expectations,1,7644
<NA>,<NA>,4726


In [31]:
assert df_clean["performance_rating"].isna().sum() == 4726

assert df_clean["performance_rating"].dropna().isin(
    ordinal_mapping.keys()
).all()

assert df_clean["performance_rating_ordinal"].dropna().isin([1, 2, 3]).all()

assert (
    df_clean["performance_rating"].isna()
    == df_clean["performance_rating_ordinal"].isna()
).all()

print("Performance rating validation passed.")

Performance rating validation passed.


### 6.5 Performance Review Score

There are 14,758 missing performance review scores.

I'll use Random Forest to estimate these values based on employee information.

The model will be tested on known scores before generating predictions. Original values will remain unchanged.

In [32]:
rating_summary = (
    df_clean.groupby("performance_rating")["performance_review_score"]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .round(3)
)

display(rating_summary)

,count,mean,median,std,min,max
performance_rating,,,,,,
Below Expectations,6476,3.400,3.4,0.860,1.0,5.0
Exceeds Expectations,25745,3.387,3.4,0.870,1.0,5.0
Meets Expectations,44021,3.390,3.4,0.868,1.0,5.0


#### 6.5.1 Model Training

I'll use employee information such as age, experience, job title, department, and performance rating to predict review scores.

In [33]:
numerical_features = [
    "age",
    "years_experience",
    "tenure_years",
    "performance_rating_ordinal"
]

categorical_features = [
    "department",
    "job_title",
    "education_level",
    "remote_status",
    "country",
    "source_dataset"
]

features = numerical_features + categorical_features
target = "performance_review_score"

known = df_clean.dropna(subset=[target]).copy()

X = known[features].copy()
y = known[target]

X[numerical_features] = X[numerical_features].astype(float)
X[categorical_features] = (
    X[categorical_features]
    .astype("object")
    .where(X[categorical_features].notna(), np.nan)
)

print(f"Available records: {len(known):,}")
print(f"Missing scores: {df_clean[target].isna().sum():,}")

Available records: 80,242
Missing scores: 14,758


In [34]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print(f"Training records: {len(X_train):,}")
print(f"Test records: {len(X_test):,}")

Training records: 64,193
Test records: 16,049


In [35]:
preprocessor = ColumnTransformer([
    (
        "numerical",
        SimpleImputer(strategy="median"),
        numerical_features
    ),
    (
        "categorical",
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]),
        categorical_features
    )
])

rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=100,
        max_depth=12,
        min_samples_leaf=10,
        random_state=42,
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)

#### 6.5.2 Model Evaluation

I'll evaluate the model using MAE and RMSE to understand how much its predictions differ from actual scores.

In [36]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"MAE: {mae:.4f}")
print(f"RMSE: {rmse:.4f}")

comparison = pd.DataFrame({
    "actual": y_test.to_numpy(),
    "predicted": np.round(y_pred, 2)
})

display(comparison.head(20))

MAE: 0.7044
RMSE: 0.8687


,actual,predicted
0,2.6,3.39
1,3.0,3.39
2,3.0,3.42
3,4.5,3.36
4,3.6,3.36
5,3.4,3.39
6,4.1,3.39
7,4.1,3.41
8,3.8,3.39
9,2.9,3.38


#### 6.5.3 Generate Estimates

I'll retrain the model using all available review scores and generate predictions for the 14,758 missing records.

Predictions will be stored in a separate column, without changing the original scores.

In [37]:
rf_model.fit(X, y)

missing_mask = df_clean[target].isna()

X_missing = df_clean.loc[missing_mask, features].copy()

X_missing[numerical_features] = (
    X_missing[numerical_features].astype(float)
)

X_missing[categorical_features] = (
    X_missing[categorical_features]
    .astype("object")
    .where(X_missing[categorical_features].notna(), np.nan)
)

predictions = rf_model.predict(X_missing)

df_clean["performance_review_score_predicted"] = np.nan

df_clean.loc[
    missing_mask,
    "performance_review_score_predicted"
] = np.round(predictions, 2)

print(f"Missing scores: {missing_mask.sum():,}")
print(f"Predicted scores: {df_clean['performance_review_score_predicted'].notna().sum():,}")

display(
    df_clean.loc[
        missing_mask,
        [
            "employee_id",
            "performance_rating",
            "performance_review_score",
            "performance_review_score_predicted",
            "performance_review_score_is_missing"
        ]
    ].head(20)
)

Missing scores: 14,758
Predicted scores: 14,758


,employee_id,performance_rating,performance_review_score,performance_review_score_predicted,performance_review_score_is_missing
2,E155936,Meets Expectations,NaN,3.40,True
8,E180678,Meets Expectations,NaN,3.40,True
13,E126956,Meets Expectations,NaN,3.38,True
22,E186746,Exceeds Expectations,NaN,3.39,True
31,E133100,Exceeds Expectations,NaN,3.39,True
34,E169811,Exceeds Expectations,NaN,3.39,True
43,E171391,Meets Expectations,NaN,3.33,True
45,E101938,Meets Expectations,NaN,3.37,True
47,E113152,Exceeds Expectations,NaN,3.37,True
54,E122383,Exceeds Expectations,NaN,3.37,True


#### 6.5.4 Validation

I'll verify that predictions were created only for missing scores, all existing scores were preserved, and the predicted values are within the valid range.

In [38]:
assert df_clean[target].equals(df[target])

assert df_clean.loc[
    ~missing_mask,
    "performance_review_score_predicted"
].isna().all()

assert df_clean.loc[
    missing_mask,
    "performance_review_score_predicted"
].notna().all()

assert df_clean[
    "performance_review_score_predicted"
].dropna().between(1, 5).all()

assert df_clean["performance_review_score_is_missing"].equals(
    df[target].isna()
)

print("Performance score prediction validation passed.")

Performance score prediction validation passed.


#### 6.5.5 Findings

Random Forest generated 14,758 estimates (MAE: 0.70).

However, predictions are heavily concentrated around 3.4, showing limited predictive power.

**Decision:** Keep the estimates separate from actual scores and flag them as low-confidence predictions.

### 6.6 Remote Status

I'll standardize the remote work categories and test whether missing values can be predicted from employee information.

In [39]:
display(
    df_clean["remote_status"]
    .value_counts(dropna=False)
    .to_frame("count")
)

,count
remote_status,
Onsite,10704
remote,10680
Hybrid,10602
On-site,10591
NaN,10585
REMOTE,10525
Remote,10481
hybrid,10462
On site,10370


#### 6.6.1 Standardize Categories

I'll merge the eight existing labels into three categories: Remote, Hybrid, and Onsite.

In [40]:
remote_mapping = {
    "remote": "Remote",
    "hybrid": "Hybrid",
    "onsite": "Onsite",
    "on-site": "Onsite",
    "on site": "Onsite"
}

normalized = (
    df_clean["remote_status"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

unknown = normalized.notna() & ~normalized.isin(remote_mapping)

assert not unknown.any(), "Unexpected remote status categories found."

df_clean["remote_status"] = normalized.map(remote_mapping).astype("string")

display(
    df_clean["remote_status"]
    .value_counts(dropna=False)
    .to_frame("count")
)

,count
remote_status,
Remote,31686
Onsite,31665
Hybrid,21064
<NA>,10585


In [41]:
assert df_clean["remote_status"].dropna().isin(
    ["Remote", "Hybrid", "Onsite"]
).all()

assert df_clean["remote_status"].isna().sum() == 10585

assert df_clean["remote_status_is_missing"].equals(
    df["remote_status"].isna()
)

print("Remote status validation passed.")

Remote status validation passed.


#### 6.6.2 Random Forest Classification

I'll use employee and job information to predict missing remote work status.

In [42]:
numerical_features = [
    "age",
    "years_experience",
    "tenure_years"
]

categorical_features = [
    "department",
    "job_title",
    "country",
    "education_level",
    "source_dataset",
    "performance_rating"
]

features = numerical_features + categorical_features
target = "remote_status"

known = df_clean.dropna(subset=[target]).copy()

X = known[features].copy()
y = known[target]

X[numerical_features] = X[numerical_features].astype(float)

X[categorical_features] = (
    X[categorical_features]
    .astype("object")
    .where(X[categorical_features].notna(), np.nan)
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Training records: {len(X_train):,}")
print(f"Test records: {len(X_test):,}")

Training records: 67,532
Test records: 16,883


In [43]:
preprocessor = ColumnTransformer([
    (
        "numerical",
        SimpleImputer(strategy="median"),
        numerical_features
    ),
    (
        "categorical",
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]),
        categorical_features
    )
])

remote_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=100,
        max_depth=12,
        min_samples_leaf=10,
        random_state=42,
        n_jobs=-1
    ))
])

remote_model.fit(X_train, y_train)

y_pred = remote_model.predict(X_test)

#### 6.6.3 Model Evaluation

I'll evaluate the model using Accuracy and Macro F1-score.

In [44]:
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average="macro")

majority_baseline = y_train.value_counts(normalize=True).max()

print(f"Accuracy: {accuracy:.2%}")
print(f"Macro F1: {f1:.4f}")
print(f"Majority-class baseline: {majority_baseline:.2%}")

print(classification_report(y_test, y_pred, zero_division=0))

Accuracy: 37.35%
Macro F1: 0.2843
Majority-class baseline: 37.54%
              precision    recall  f1-score   support

      Hybrid       0.00      0.00      0.00      4213
      Onsite       0.37      0.47      0.42      6333
      Remote       0.37      0.53      0.44      6337

    accuracy                           0.37     16883
   macro avg       0.25      0.33      0.28     16883
weighted avg       0.28      0.37      0.32     16883



#### 6.6.4 Findings

Random Forest achieved 37.35% accuracy, below the 37.54% majority-class baseline.

The model also failed to identify any Hybrid employees.

**Decision:** Keep 10,585 missing values unchanged. The available features do not support reliable predictions.

### 6.7 Education Level

I'll standardize education levels and test whether missing values can be predicted from employee information.

In [45]:
display(
    df_clean["education_level"]
    .value_counts(dropna=False)
    .to_frame("count")
)

,count
education_level,
PhD,9188
bachelor's degree,9158
Master's,9117
High School,9083
PhD.,9050
MSc,9022
BSc,9006
Bachelors,8998
Bachelor's,8873


#### 6.7.1 Standardize Categories

I'll merge the 10 education labels into 5 categories.

In [46]:
education_mapping = {
    "high school": "High School",
    "associate degree": "Associate Degree",
    "bachelor's": "Bachelor's",
    "bachelors": "Bachelor's",
    "bachelor's degree": "Bachelor's",
    "bsc": "Bachelor's",
    "master's": "Master's",
    "msc": "Master's",
    "phd": "PhD",
    "phd.": "PhD"
}

normalized = (
    df_clean["education_level"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

unknown = normalized.notna() & ~normalized.isin(education_mapping)

assert not unknown.any(), "Unexpected education categories found."

df_clean["education_level"] = normalized.map(education_mapping).astype("string")

display(
    df_clean["education_level"]
    .value_counts(dropna=False)
    .to_frame("count")
)

,count
education_level,
Bachelor's,36035
PhD,18238
Master's,18139
High School,9083
Associate Degree,8849
<NA>,4656


#### 6.7.2 Random Forest Classification

I'll use employee and job information to predict missing education levels.

In [47]:
numerical_features = [
    "age",
    "years_experience",
    "tenure_years"
]

categorical_features = [
    "department",
    "job_title",
    "country",
    "source_dataset",
    "remote_status",
    "performance_rating"
]

features = numerical_features + categorical_features
target = "education_level"

known = df_clean.dropna(subset=[target]).copy()

X = known[features].copy()
y = known[target]

X[numerical_features] = X[numerical_features].astype(float)

X[categorical_features] = (
    X[categorical_features]
    .astype("object")
    .where(X[categorical_features].notna(), np.nan)
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Training records: {len(X_train):,}")
print(f"Test records: {len(X_test):,}")

Training records: 72,275
Test records: 18,069


In [48]:
preprocessor = ColumnTransformer([
    (
        "numerical",
        SimpleImputer(strategy="median"),
        numerical_features
    ),
    (
        "categorical",
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]),
        categorical_features
    )
])

education_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=100,
        max_depth=12,
        min_samples_leaf=10,
        random_state=42,
        n_jobs=-1
    ))
])

education_model.fit(X_train, y_train)

y_pred = education_model.predict(X_test)

In [49]:
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average="macro")

majority_baseline = y_train.value_counts(normalize=True).max()

print(f"Accuracy: {accuracy:.2%}")
print(f"Macro F1: {f1:.4f}")
print(f"Majority-class baseline: {majority_baseline:.2%}")

print(classification_report(y_test, y_pred, zero_division=0))

Accuracy: 39.89%
Macro F1: 0.1141
Majority-class baseline: 39.89%
                  precision    recall  f1-score   support

Associate Degree       0.00      0.00      0.00      1770
      Bachelor's       0.40      1.00      0.57      7207
     High School       0.00      0.00      0.00      1816
        Master's       0.00      0.00      0.00      3628
             PhD       0.00      0.00      0.00      3648

        accuracy                           0.40     18069
       macro avg       0.08      0.20      0.11     18069
    weighted avg       0.16      0.40      0.23     18069



#### 6.7.3 KNN Classification

Random Forest predicted only the majority class (Bachelor's).

I'll test KNN to see if similar employee records can provide better predictions.

In [50]:
knn_preprocessor = ColumnTransformer([
    (
        "numerical",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]),
        numerical_features
    ),
    (
        "categorical",
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]),
        categorical_features
    )
])

education_knn = Pipeline([
    ("preprocessor", knn_preprocessor),
    ("model", KNeighborsClassifier(
        n_neighbors=25,
        weights="distance",
        n_jobs=-1
    ))
])

education_knn.fit(X_train, y_train)

knn_pred = education_knn.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, knn_pred):.2%}")
print(f"Macro F1: {f1_score(y_test, knn_pred, average='macro'):.4f}")

print(classification_report(y_test, knn_pred, zero_division=0))

Accuracy: 35.74%
Macro F1: 0.1558
                  precision    recall  f1-score   support

Associate Degree       0.07      0.00      0.01      1770
      Bachelor's       0.40      0.81      0.53      7207
     High School       0.07      0.00      0.01      1816
        Master's       0.18      0.08      0.11      3628
             PhD       0.20      0.09      0.12      3648

        accuracy                           0.36     18069
       macro avg       0.18      0.20      0.16     18069
    weighted avg       0.25      0.36      0.26     18069



#### 6.7.4 Findings

Random Forest achieved 39.89% accuracy, matching the majority-class baseline.

KNN achieved 35.74% accuracy and a slightly higher Macro F1 score (0.1558).

Neither model provided reliable predictions.

**Decision:** Keep 4,656 missing education values unchanged.

#### 6.7.5 Validation

In [51]:
assert df_clean["education_level"].isna().sum() == 4656

assert df_clean["education_level"].dropna().isin(
    education_mapping.values()
).all()

assert df_clean["education_level_is_missing"].equals(
    df["education_level"].isna()
)

print("Education level validation passed.")

Education level validation passed.


### 6.8 Gender and Ethnicity

Missing gender and ethnicity values will not be predicted.

These are sensitive demographic attributes, and assigning unverified values could introduce bias.

All 2,865 missing gender values and 23,796 missing ethnicity values were preserved.

## 7. Export

Save the processed dataset for the next cleaning stage.

In [52]:
from pathlib import Path

output_path = Path("data/processed/02_missing_values_resolved.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

df_clean.to_csv(output_path, index=False)

print(f"Saved: {output_path}")
print(f"Rows: {len(df_clean):,}")
print(f"Columns: {len(df_clean.columns)}")

Saved: data/processed/02_missing_values_resolved.csv
Rows: 95,000
Columns: 32


## 8. Summary

### Data Recovery

- **931** missing tenure values recovered from hire dates.
- Existing tenure values preserved.

### Predictions

- **2,839** missing ages estimated using the median method.
- **899** missing experience values estimated using KNN.
- **14,758** performance review scores estimated using Random Forest.

All predictions were stored in separate columns.

### Standardization

- Performance rating labels standardized.
- Remote work categories standardized.
- Education level categories standardized.
- Nine missing-value flags created.

### Preserved Missing Values

- **10,585** remote status values.
- **4,656** education values.
- **2,865** gender values.
- **23,796** ethnicity values.

Unreliable estimates were not used to replace original values.

### Final Dataset

- **95,000** employee records.
- **32** columns.
- Ready for the next cleaning stage.